In [1]:
import pandas as pd
import numpy as np
import re

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from lightgbm import LGBMRegressor

In [2]:
from sklearn.decomposition import PCA

In [3]:
train_df = pd.read_csv(
    "/content/train.csv"
)

train_df.shape

(75000, 4)

In [4]:
sample_df = train_df.sample(
    20000,
    random_state=42
).reset_index(drop=True)

sample_df.shape

(20000, 4)

In [5]:
X_train_text, X_valid_text, y_train, y_valid = train_test_split(
    sample_df["catalog_content"],
    sample_df["price"],
    test_size=0.2,
    random_state=42
)

In [6]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(
    X_train_text
)

X_valid_tfidf = tfidf.transform(
    X_valid_text
)

print(X_train_tfidf.shape)
print(X_valid_tfidf.shape)

(16000, 5000)
(4000, 5000)


In [7]:
def extract_quantity_features(text):
    text = str(text).lower()

    features = {}

    patterns = {
        "ounce_feature": r'(\d+(?:\.\d+)?)\s*(?:oz|ounce|ounces)\b',
        "pound_feature": r'(\d+(?:\.\d+)?)\s*(?:lb|lbs|pound|pounds)\b',
        "gram_feature": r'(\d+(?:\.\d+)?)\s*(?:g|gram|grams)\b',
        "kg_feature": r'(\d+(?:\.\d+)?)\s*(?:kg|kilogram|kilograms)\b',
        "ml_feature": r'(\d+(?:\.\d+)?)\s*(?:ml|milliliter|milliliters)\b',
        "liter_feature": r'(\d+(?:\.\d+)?)\s*(?:l|liter|liters)\b',

        "pack_feature": r'pack of\s*(\d+)|(\d+)[-\s]*pack\b',
        "count_feature": r'(\d+)\s*(?:count|ct)\b',
        "serving_feature": r'(\d+)\s*(?:servings?)\b',
        "bottle_feature": r'(\d+)\s*bottles?\b',
        "bag_feature": r'(\d+)\s*bags?\b',
        "case_feature": r'(\d+)\s*case\b',
        "dozen_feature": r'(\d+)\s*dozen\b'
    }

    for feature_name, pattern in patterns.items():
        matches = re.findall(pattern, text)

        numbers = []

        for match in matches:
            if isinstance(match, tuple):
                match = [m for m in match if m != ""]
                if len(match) > 0:
                    numbers.extend(match)
            else:
                numbers.append(match)

        numbers = [
            float(x)
            for x in numbers
            if x != "" and float(x) < 1000
        ]

        features[feature_name] = max(numbers) if numbers else 0

    return pd.Series(features)

In [ ]:
quantity_features = train_df[
    "catalog_content"
].apply(extract_quantity_features)

quantity_features.head()

In [ ]:
quantity_features.describe()

In [10]:
quantity_train = quantity_features.loc[
    X_train_text.index
]

quantity_valid = quantity_features.loc[
    X_valid_text.index
]

print(quantity_train.shape)
print(quantity_valid.shape)

(16000, 13)
(4000, 13)


In [11]:
quantity_train_sparse = csr_matrix(
    quantity_train.values
)

quantity_valid_sparse = csr_matrix(
    quantity_valid.values
)

In [12]:
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import torch
import timm

from torchvision import transforms

In [ ]:
model = timm.create_model(
    "resnet50",
    pretrained=True,
    num_classes=0
)

model.eval()

In [14]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [15]:
def get_image_embedding(url):

    try:
        response = requests.get(
            url,
            timeout=5
        )

        image = Image.open(
            BytesIO(response.content)
        ).convert("RGB")

        image = transform(image)

        image = image.unsqueeze(0)

        with torch.no_grad():

            embedding = model(
                image
            ).cpu().numpy()

        return embedding.squeeze()

    except Exception:

        return np.zeros(
            2048,
            dtype=np.float32
        )

In [17]:
X_train_images = sample_df.loc[
    X_train_text.index,
    "image_link"
]

X_valid_images = sample_df.loc[
    X_valid_text.index,
    "image_link"
]

In [ ]:
import numpy as np
import os
from tqdm import tqdm

save_folder = "saved_embeddings"

os.makedirs(
    save_folder,
    exist_ok=True
)

batch_size = 500

for start_idx in range(
    0,
    len(X_train_images),
    batch_size
):

    end_idx = min(
        start_idx + batch_size,
        len(X_train_images)
    )

    batch_urls = X_train_images.iloc[
        start_idx:end_idx
    ]

    batch_embeddings = []

    for url in tqdm(batch_urls):

        embedding = get_image_embedding(
            url
        )

        batch_embeddings.append(
            embedding
        )

    batch_embeddings = np.array(
        batch_embeddings,
        dtype=np.float32
    )

    np.save(
        f"{save_folder}/train_{start_idx}_{end_idx}.npy",
        batch_embeddings
    )

    print(
        f"Saved train batch: {start_idx} to {end_idx}"
    )

In [ ]:
batch_size = 500

for start_idx in range(
    0,
    len(X_valid_images),
    batch_size
):

    end_idx = min(
        start_idx + batch_size,
        len(X_valid_images)
    )

    batch_urls = X_valid_images.iloc[
        start_idx:end_idx
    ]

    batch_embeddings = []

    for url in tqdm(batch_urls):

        embedding = get_image_embedding(
            url
        )

        batch_embeddings.append(
            embedding
        )

    batch_embeddings = np.array(
        batch_embeddings,
        dtype=np.float32
    )

    np.save(
        f"{save_folder}/valid_{start_idx}_{end_idx}.npy",
        batch_embeddings
    )

    print(
        f"Saved valid batch: {start_idx} to {end_idx}"
    )

In [ ]:
train_files = sorted(
    [
        f for f in os.listdir("saved_embeddings")
        if f.startswith("train")
    ],
    key=lambda x: int(
        x.split("_")[1]
    )
)

train_embeddings = []

for file in train_files:

    path = os.path.join(
        "saved_embeddings",
        file
    )

    batch = np.load(path)

    train_embeddings.append(
        batch
    )

train_embeddings = np.vstack(
    train_embeddings
)

print(
    train_embeddings.shape
)

In [ ]:
valid_files = sorted(
    [
        f for f in os.listdir("saved_embeddings")
        if f.startswith("valid")
    ],
    key=lambda x: int(
        x.split("_")[1]
    )
)

valid_embeddings = []

for file in valid_files:

    path = os.path.join(
        "saved_embeddings",
        file
    )

    batch = np.load(path)

    valid_embeddings.append(
        batch
    )

valid_embeddings = np.vstack(
    valid_embeddings
)

print(
    valid_embeddings.shape
)

In [ ]:
pca = PCA(n_components=500)

train_embeddings_pca = pca.fit_transform(
    train_embeddings
)

valid_embeddings_pca = pca.transform(
    valid_embeddings
)

print(train_embeddings_pca.shape)
print(valid_embeddings_pca.shape)

train_image_embeddings = csr_matrix(
    train_embeddings_pca
)

valid_image_embeddings = csr_matrix(
    valid_embeddings_pca
)

In [ ]:
X_train_final = hstack([
    X_train_tfidf,
    quantity_train_sparse,
    train_image_embeddings
])

X_valid_final = hstack([
    X_valid_tfidf,
    quantity_valid_sparse,
    valid_image_embeddings
])

print(X_train_final.shape)
print(X_valid_final.shape)

In [ ]:
multimodal_model = LGBMRegressor(
    n_estimators=1500,
    learning_rate=0.02,
    num_leaves=100,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=0.2,
    random_state=42
)
y_train_log = np.log1p(y_train)

multimodal_model.fit(
    X_train_final,
    y_train_log
)

In [ ]:
multimodal_predictions = np.expm1(
    multimodal_model.predict(
        X_valid_final
    )
)

In [29]:
mae = mean_absolute_error(
    y_valid,
    multimodal_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_valid,
        multimodal_predictions
    )
)

r2 = r2_score(
    y_valid,
    multimodal_predictions
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 12.189997035718966
RMSE: 24.946705168669403
R2 Score: 0.30871883700184033
